<a href="https://colab.research.google.com/github/Faizan-Rashid/facial-emotion-recognition-deep-learning/blob/main/helper_functions_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**This is the notebook that having functions that might not be present in [`helper_functions.py`](https://github.com/Faizan-Rashid/facial-emotion-recognition-deep-learning/blob/main/helper_functions.py)**

## 1. Reusable Functions

### get_new_mobilenet_v3_instance function to create new instance

In [ ]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

def get_new_mobilenet_v3_instance(device: torch.device=device):
  model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
  # Modify final layer for 7 FER-2013 emotion classes
  model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, 7)
  model = model.to(device)
  return model

### Training and Testing loop (functions)

In [ ]:
from tqdm.auto import tqdm

# training function
def train_step(model: torch.nn.Module,
               loss_fn: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device=device):
  """Perform training on model trying to learn on data_loader"""
  ### Training
  model.train()
  loss, accuracy = 0, 0

  # reset the accumulated accuracy per batch
  accuracy_fn.reset()
  for (X, y) in tqdm(data_loader):
    # Put the data on target device
    X, y = X.to(device), y.to(device)

    # 1. Forward pass (give raw logits)
    y_preds = model(X)

    # 2. calculate the loss and accuracy per batch
    train_loss_batch = loss_fn(y_preds, y)
    loss += train_loss_batch.item()

    accuracy_fn.update((y_preds, y))
    # acc = accuracy_fn(y_true=y, y_pred=y_preds.argmax(dim=1))
    # accuracy += acc

    # 3. optimizer zero grad
    optimizer.zero_grad()

    # 4. perform back propogation
    train_loss_batch.backward()

    # 5. optimizer step (gradient decent)
    optimizer.step()

  # calculate average loss and accuracy
  loss /= len(data_loader)
  # accuracy /= len(data_loader)
  accuracy = accuracy_fn.compute()

  # Print
  print(f"Train loss: {loss:.4f} | train accuracy: {accuracy:.4f}")

  # return values (accuracy and loss) for results to be saved in history dictionary
  return (loss, accuracy)

In [ ]:
# test function
def test_step(model: torch.nn.Module,
              loss_fn: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              accuracy_fn,
              device: torch.device=device):
  """Perform testing on model going over data_loader"""

  ### Testing
  test_loss, test_acc = 0, 0
  accuracy_fn.reset()
  model.eval()
  with torch.inference_mode():

    # reset the accumulated accuracy per batch
    accuracy_fn.reset()
    for X_test, y_test in tqdm(data_loader):
      # Put the data on target device
      X_test, y_test = X_test.to(device), y_test.to(device)
      # 1. Forward pass
      y_test_preds = model(X_test)
      # 2. Calculate the loss
      test_loss_batch = loss_fn(y_test_preds, y_test)
      test_loss += test_loss_batch
      # 3. Calculate the accuracy
      accuracy_fn.update((y_test_preds, y_test))
      # test_acc_batch = accuracy_fn(y_true=y_test, y_pred=y_test_preds.argmax(dim=1))
      # test_acc += test_acc_batch

    # calculate the average accuracy and test loss
    test_loss /= len(data_loader)
    # test_acc /= len(data_loader)
    test_acc = accuracy_fn.compute()

  # Print what's happening
  print(f"Test loss: {test_loss:.4f} | test accuracy: {test_acc:.4f}")

  # return values (accuracy and loss) for results to be saved in history dictionary
  return (test_loss, test_acc)


In [ ]:
# more optimized

from tqdm.auto import tqdm
import torch

def train_step_1(model: torch.nn.Module,
               loss_fn: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device):
    """Perform training on model trying to learn on data_loader"""
    model.train()
    loss = 0

    # Reset metric accumulator
    accuracy_fn.reset()

    for (X, y) in tqdm(data_loader, desc="Training Batches", leave=False):
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        outputs = model(X)

        # FIX: Dynamically extract logits if using a Hugging Face Transformer
        y_preds = outputs.logits if hasattr(outputs, "logits") else outputs

        # 2. Calculate loss and update metrics
        train_loss_batch = loss_fn(y_preds, y)
        loss += train_loss_batch.item()

        # Update running accuracy metrics
        accuracy_fn.update((y_preds, y))

        # 3. Backward pass and optimization steps
        optimizer.zero_grad()
        train_loss_batch.backward()
        optimizer.step()

    # Calculate averages
    loss /= len(data_loader)
    accuracy = accuracy_fn.compute()

    print(f"Train loss: {loss:.4f} | train accuracy: {accuracy:.4f}")
    return (loss, accuracy)

def test_step_1(model: torch.nn.Module,
              loss_fn: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              accuracy_fn,
              device: torch.device):
    """Perform testing on model going over data_loader"""
    test_loss = 0
    accuracy_fn.reset()
    model.eval()

    with torch.inference_mode():
        for X_test, y_test in tqdm(data_loader, desc="Testing Batches", leave=False):
            X_test, y_test = X_test.to(device), y_test.to(device)

            # 1. Forward pass
            outputs = model(X_test)

            # FIX: Dynamically extract logits if using a Hugging Face Transformer
            y_test_preds = outputs.logits if hasattr(outputs, "logits") else outputs

            # 2. Calculate batch loss (Fix: convert item to float early)
            test_loss_batch = loss_fn(y_test_preds, y_test)
            test_loss += test_loss_batch.item()

            # 3. Track metrics
            accuracy_fn.update((y_test_preds, y_test))

        # Calculate averages
        test_loss /= len(data_loader)
        test_acc = accuracy_fn.compute()

    print(f"Test loss: {test_loss:.4f} | test accuracy: {test_acc:.4f}")
    return (test_loss, test_acc)


### function for Timimg our training

In [ ]:
# Function to measure the time of experiments
from timeit import default_timer as time

def print_train_time(
    start: float,
    end: float,
    device:torch.device=None
):
  """Prints difference between start and end time"""

  total_time = end - start
  print(f"Train time for device {device}: {total_time}")
  return total_time

### function for evaluating the model

In [ ]:

from timeit import default_timer as timer
from tqdm.auto import tqdm

def eval_model(
    model: torch.nn.Module,
    data_loader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    accuracy_fn,
):
  """evaluates the given model - returns a dictionary containing results of model's prediction on data loader

  Args:
        model (torch.nn.Module): A PyTorch model capable of making predictions on data_loader.
        data_loader (torch.utils.data.DataLoader): The target dataset to predict on.
        loss_fn (torch.nn.Module): The loss function of model.
        accuracy_fn: An accuracy function to compare the models predictions to the truth labels.

  Returns:
        (dict): Results of model making predictions on data_loader."""

  model.eval()
  accuracy_fn.reset()
  with torch.inference_mode():
    loss, accuracy = 0, 0
    for X, y in tqdm(data_loader):
      X = X.to(device)
      y = y.to(device)
      y_preds = model(X)
      # calculate and add loss value
      loss_batch = loss_fn(y_preds, y)
      loss += loss_batch
      # calculate and add accuracy value
      acc = accuracy_fn.update((y_preds, y))
      # acc = accuracy_fn(y_true=y, y_pred=y_preds.argmax(dim=1))
      # accuracy += acc

    # calculate the avergae loss and accuracy
    loss /= len(data_loader)
    # accuracy /= len(data_loader)
    accuracy = accuracy_fn.compute()

  return {
      "model_name": model.__class__.__name__,
      "loss":loss,
      "accuracy":accuracy
  }

In [ ]:
from timeit import default_timer as timer
from tqdm.auto import tqdm

def eval_model_1(
    model: torch.nn.Module,
    data_loader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    accuracy_fn,
):
  """evaluates the given model - returns a dictionary containing results of model's prediction on data loader

  Args:
        model (torch.nn.Module): A PyTorch model capable of making predictions on data_loader.
        data_loader (torch.utils.data.DataLoader): The target dataset to predict on.
        loss_fn (torch.nn.Module): The loss function of model.
        accuracy_fn: An accuracy function to compare the models predictions to the truth labels.

  Returns:
        (dict): Results of model making predictions on data_loader."""

  model.eval()
  accuracy_fn.reset()
  with torch.inference_mode():
    loss, accuracy = 0, 0
    for X, y in tqdm(data_loader):
      X = X.to(device)
      y = y.to(device)
      y_preds = model(X)

      # Extract logits if y_preds is an ImageClassifierOutput object (common for HuggingFace models)
      if hasattr(y_preds, 'logits'):
          logits = y_preds.logits
      else:
          logits = y_preds

      # calculate and add loss value
      loss_batch = loss_fn(logits, y)
      loss += loss_batch
      # calculate and add accuracy value
      acc = accuracy_fn.update((logits, y))

    # calculate the avergae loss and accuracy
    loss /= len(data_loader)
    accuracy = accuracy_fn.compute()

  return {
      "model_name": model.__class__.__name__,
      "loss":loss,
      "accuracy":accuracy
  }

### Accuracy function from pytorch-ignite

In [ ]:
!pip install pytorch-ignite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.8/371.8 kB 1.7 MB/s eta 0:00:00


In [ ]:
from ignite.metrics import Accuracy

accuracy_fn = Accuracy()

### function for counting parameters of model

In [ ]:
def params_count(model):
  total_params = sum(p.numel() for p in model.parameters())
  trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

  print(f"Total parameters: {total_params/1e6:.3f}M")
  print(f"Trainable parameters: {trainable_params/1e6:.3f}M")

  return total_params

### Training our model

#### `train_give_results` function

In [ ]:
# train_give_results function with learning rate scheduler
torch.manual_seed(42)

from typing import Callable
from timeit import default_timer as timer
from tqdm.auto import tqdm


def train_give_results(model: torch.nn.Module,
                       loss_fn: torch.nn.Module,
                       optimizer: torch.optim.Optimizer,
                       accuracy_fn: Callable,
                       train_dataloader: torch.utils.data.DataLoader,
                       test_dataloader: torch.utils.data.DataLoader,
                       train_step: Callable,
                       test_step: Callable,
                       print_train_time: Callable,
                       lr_scheduler: Callable, # To apply lr scheduling
                       patience: int, # To apply Early stopping
                       epochs: int,
                       device: torch.device=device):

  ### INITIALIZE RESULTS DICTIONARY
  training_results = {
      "results": [],
      "train_time":0,
      "model_layers":None,
      "device": device
  }

  ### MEASURE TIME BEFORE TRAINING
  time_before_train = timer()


  # variables for early stopping
  patience_counter = 0
  best_loss = float("inf")

  for epoch in tqdm(range(epochs)):
    print(f"epoch: {epoch}--------------")

    ### 1. Training
    (train_loss, train_acc) = train_step(model=model,
              loss_fn=loss_fn,
              optimizer=optimizer,
              accuracy_fn=accuracy_fn,
              data_loader=train_dataloader,
              device=device)

    ### 2. Testing
    (test_loss, test_acc) = test_step(model=model,
              loss_fn=loss_fn,
              data_loader=test_dataloader,
              accuracy_fn=accuracy_fn,
              device=device)

    ### 3. Apply lr scheduler
    lr_scheduler.step(test_loss)

    ### 4. Apply early stopping
    if(test_loss < best_loss):
      best_loss = test_loss
      patience_counter = 0
      torch.save(model.state_dict(), "best_model.pt")
    else:
      patience_counter += 1

    if (patience_counter >= patience):
      print("Early stopping triggered")
      break

    ### 5. SAVE RESULTS
    training_results['results'].append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
    })

  ### MEASURE TIME AFTR TRAINING
  time_after_train = timer()

  ### MEASURE TIME TAKEN
  train_time = print_train_time(time_before_train, time_after_train, device="cuda")

  ### SAVE RESULTS
  training_results["train_time"] = train_time
  training_results["device"] = device
  training_results["model_layers"] = model.features

  ### RETURN RESULTS
  return training_results

In [ ]:
# More optimized

from typing import Callable
from timeit import default_timer as timer

def train_give_results_1(model: torch.nn.Module,
                       loss_fn: torch.nn.Module,
                       optimizer: torch.optim.Optimizer,
                       accuracy_fn: Callable,
                       train_dataloader: torch.utils.data.DataLoader,
                       test_dataloader: torch.utils.data.DataLoader,
                       train_step: Callable,
                       test_step: Callable,
                       print_train_time: Callable,
                       lr_scheduler: Callable,
                       patience: int,
                       epochs: int,
                       device: torch.device):

    ### INITIALIZE RESULTS DICTIONARY
    training_results = {
        "results": [],
        "train_time": 0,
        "model_layers": None,
        "device": device
    }

    ### MEASURE TIME BEFORE TRAINING
    time_before_train = timer()

    # Early stopping trackers
    patience_counter = 0
    best_loss = float("inf")

    for epoch in tqdm(range(epochs), desc="Total Epochs"):
        print(f"\nepoch: {epoch}--------------")

        ### 1. Training Step
        train_loss, train_acc = train_step(
            model=model,
            loss_fn=loss_fn,
            optimizer=optimizer,
            accuracy_fn=accuracy_fn,
            data_loader=train_dataloader,
            device=device
        )

        ### 2. Testing Step
        test_loss, test_acc = test_step(
            model=model,
            loss_fn=loss_fn,
            data_loader=test_dataloader,
            accuracy_fn=accuracy_fn,
            device=device
        )

        # FIX: Ensure metrics are pulled out of tensors for calculations/logging
        if isinstance(train_acc, torch.Tensor): train_acc = train_acc.item()
        if isinstance(test_acc, torch.Tensor): test_acc = test_acc.item()

        ### 3. Apply lr scheduler
        lr_scheduler.step(test_loss)

        ### 4. Apply early stopping
        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            # Save complete state dictionary
            torch.save(model.state_dict(), "best_model.pt")
            print(f"🔥 New best model saved with test loss: {test_loss:.4f}")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"🛑 Early stopping triggered at epoch {epoch}")
            break

        ### 5. SAVE RESULTS
        training_results['results'].append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_loss,
            "test_acc": test_acc,
        })

    ### MEASURE TIME AFTER TRAINING
    time_after_train = timer()
    train_time = print_train_time(time_before_train, time_after_train, device="cuda")

    ### SAVE METADATA
    training_results["train_time"] = train_time
    training_results["device"] = device

    # FIX: Dynamic naming fallback so Transformers don't break on missing '.features' attribute
    if hasattr(model, "features"):
        training_results["model_layers"] = model.features
    else:
        training_results["model_layers"] = str(type(model))

    return training_results

#### `train_give_results_sc` function

changed train_give_results to for `CosineAnnealingWarmRestarts` learning rate schedular

In [ ]:
### WE'LL REDEFINE train_give_results because of CosineAnnealingWarmRestarts


def train_give_results_sc(model: torch.nn.Module,
                       loss_fn: torch.nn.Module,
                       optimizer: torch.optim.Optimizer,
                       accuracy_fn: Callable,
                       train_dataloader: torch.utils.data.DataLoader,
                       test_dataloader: torch.utils.data.DataLoader,
                       train_step: Callable,
                       test_step: Callable,
                       print_train_time: Callable,
                       lr_scheduler: Callable, # To apply lr scheduling
                       patience: int, # To apply Early stopping
                       epochs: int,
                       device: torch.device=device):

  ### INITIALIZE RESULTS DICTIONARY
  training_results = {
      "results": [],
      "train_time":0,
      "model_layers":None,
      "device": device
  }

  ### MEASURE TIME BEFORE TRAINING
  time_before_train = timer()


  # variables for early stopping
  patience_counter = 0
  best_loss = float("inf")

  for epoch in tqdm(range(epochs)):
    print(f"epoch: {epoch}--------------")

    ### 1. Training
    (train_loss, train_acc) = train_step(model=model,
              loss_fn=loss_fn,
              optimizer=optimizer,
              accuracy_fn=accuracy_fn,
              data_loader=train_dataloader,
              device=device)

    ### 2. Testing
    (test_loss, test_acc) = test_step(model=model,
              loss_fn=loss_fn,
              data_loader=test_dataloader,
              accuracy_fn=accuracy_fn,
              device=device)

    ### 3. Apply lr scheduler
    lr_scheduler.step()

    ### 4. SAVE RESULTS
    training_results['results'].append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
    })

    ### 5. Apply early stopping
    if(test_loss < best_loss):
      best_loss = test_loss
      patience_counter = 0
      torch.save(model.state_dict(), "best_model.pt")
    else:
      patience_counter += 1

    if (patience_counter >= patience):
      print("Early stopping triggered")
      break

  ### MEASURE TIME AFTR TRAINING
  time_after_train = timer()

  ### MEASURE TIME TAKEN
  train_time = print_train_time(time_before_train, time_after_train, device="cuda")

  ### LOAD THE BEST WEIGHTS BACK IN MEMEORY:
  model.load_state_dict(torch.load("best_model.pt", weights_only=True))

  ### SAVE RESULTS
  training_results["train_time"] = train_time
  training_results["device"] = device
  # training_results["model_layers"] = model.features

  ### RETURN RESULTS
  return training_results

#### `train_give_results_td` function *--to develop*
this is for eperiment tracking using tensorboard

In [ ]:
# # More optimized

# from typing import Callable
# from timeit import default_timer as timer

# def train_give_results_td(model: torch.nn.Module,
#                           loss_fn: torch.nn.Module,
#                           optimizer: torch.optim.Optimizer,
#                           accuracy_fn: Callable,
#                           train_dataloader: torch.utils.data.DataLoader,
#                           test_dataloader: torch.utils.data.DataLoader,
#                           train_step: Callable,
#                           test_step: Callable,
#                           print_train_time: Callable,
#                           lr_scheduler: Callable,
#                           writer: torch.utils.tensorboard.SummaryWriter, # for tracking experiment
#                           patience: int,
#                           epochs: int,
#                           device: torch.device):

#     ### INITIALIZE RESULTS DICTIONARY
#     training_results = {
#         "results": [],
#         "train_time": 0,
#         "model_layers": None,
#         "device": device
#     }

#     ### MEASURE TIME BEFORE TRAINING
#     time_before_train = timer()

#     # Early stopping trackers
#     patience_counter = 0
#     best_loss = float("inf")

#     for epoch in tqdm(range(epochs), desc="Total Epochs"):
#         print(f"\nepoch: {epoch}--------------")

#         ### 1. Training Step
#         train_loss, train_acc = train_step(
#             model=model,
#             loss_fn=loss_fn,
#             optimizer=optimizer,
#             accuracy_fn=accuracy_fn,
#             data_loader=train_dataloader,
#             device=device
#         )

#         ### 2. Testing Step
#         test_loss, test_acc = test_step(
#             model=model,
#             loss_fn=loss_fn,
#             data_loader=test_dataloader,
#             accuracy_fn=accuracy_fn,
#             device=device
#         )

#         # FIX: Ensure metrics are pulled out of tensors for calculations/logging
#         if isinstance(train_acc, torch.Tensor): train_acc = train_acc.item()
#         if isinstance(test_acc, torch.Tensor): test_acc = test_acc.item()

#         ### 3. Apply lr scheduler
#         lr_scheduler.step(test_loss)

#         ### 4. Apply early stopping
#         if test_loss < best_loss:
#             best_loss = test_loss
#             patience_counter = 0
#             # Save complete state dictionary
#             torch.save(model.state_dict(), "best_model.pt")
#             print(f"🔥 New best model saved with test loss: {test_loss:.4f}")
#         else:
#             patience_counter += 1

#         if patience_counter >= patience:
#             print(f"🛑 Early stopping triggered at epoch {epoch}")
#             break

#         ### 5. SAVE RESULTS
#         training_results['results'].append({
#             "epoch": epoch,
#             "train_loss": train_loss,
#             "train_acc": train_acc,
#             "test_loss": test_loss,
#             "test_acc": test_acc,
#         })

#         ### 6. EXPERIMENT TRACKING
#         writer.add_scalars(main_tag="Loss",
#                            tag_scalar_dict={"train_loss": train_loss,
#                                             "test_loss": test_loss},
#                            global_step=epoch)

#         writer.add_scalars(main_tag="Accuracy",
#                            tag_scalar_dict={"train_acc": train_acc,
#                                             "test_acc": test_acc},
#                            global_step=epoch)

#     writer.close()

#     ### MEASURE TIME AFTER TRAINING
#     time_after_train = timer()
#     train_time = print_train_time(time_before_train, time_after_train, device="cuda")

#     ### SAVE METADATA
#     training_results["train_time"] = train_time
#     training_results["device"] = device

#     # FIX: Dynamic naming fallback so Transformers don't break on missing '.features' attribute
#     if hasattr(model, "features"):
#         training_results["model_layers"] = model.features
#     else:
#         training_results["model_layers"] = str(type(model))

#     return training_results

### Function to save results and model state

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def save_model_results(model: torch.nn.Module,
                       optimizer: torch.optim.Optimizer | None,
                       epochs: int,
                       results: dict,
                       path: str):
  from google.colab import drive
  drive.mount('/content/drive')

  checkpoint = {
        "epochs": epochs,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "results": results
    }

  ### SAVE TO A FOLDER IN DRIVE
  torch.save(checkpoint, path)
  print("Checkpoint saved to Google Drive!")

In [ ]:
def save_model_results(model: torch.nn.Module,
                       optimizer: torch.optim.Optimizer | None,
                       epochs: int,
                       results: dict,
                       path: str):
  checkpoint = {
        "epochs": epochs,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "results": results
    }

  ### SAVE TO A FOLDER IN DRIVE
  torch.save(checkpoint, path)
  print("Checkpoint saved to Google Drive!")

### Load saved checkpoint function

In [ ]:

def load_saved_checkpoint(model: torch.nn.Module,
                          device: torch.device,
                          optimizer: torch.optim.Optimizer | None,
                          path: str,
                          load_trained_params: bool
                          ):
  checkpoint = torch.load(path, map_location=torch.device(device), weights_only=False)

  if(load_trained_params):
    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

  return checkpoint


## 3. Modular functions

### `get_fer2013_data()`
to download `FER 2013` data

In [ ]:
import kagglehub
from pathlib import Path
import requests
import zipfile
import os

def get_fer2013_data(destination: str,
                     remove_source: bool):
  """
  example usage:
  data_dir = get_fer2013_data(destination="fer2013",
                              remove_source=True)
  """

  data_path = Path("data")
  image_path = data_path / destination

  # Download
  if not image_path.is_dir():
    image_path.mkdir(parents=True, exist_ok=True)

  if not (data_path / "fer2013.zip").is_file():
    print(f"zip file not found downloading from kagglehub")
    # command to download fer2013 data from kaggle - https://www.kaggle.com/datasets/msambare/fer2013
    !kaggle datasets download msambare/fer2013 --path $data_path

    # extract the zip file data into image_path
    with zipfile.ZipFile(data_path / "fer2013.zip", "r") as zip_ref:
      zip_ref.extractall(image_path)

  if remove_source:
    # remove the zip file
    os.remove(data_path / "fer2013.zip")
    print(f"Extracted {data_path/"fer2013.zip"} file to {image_path}")

  return image_path

### `create_dataloaders()`

for getting data loaders, datasets and classes

In [ ]:
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import os

# NUM_WORKERS = os.cpu_count()

device = "cuda" if torch.cuda.is_available() else "cpu"

PIN_MEMORY = True if device == "cuda" else False

def create_dataloaders(train_dir: str,
                       test_dir: str,
                       batch_size: int,
                       train_transform: transforms.Compose=None,
                       test_transform: transforms.Compose=None,
                      #  num_workers: int=NUM_WORKERS,
                       pin_memory: bool=PIN_MEMORY):
  if train_transform is None or test_transform is None:
    print(f"[INFO] train and test transform both need to be passed. Exitting...")
    return None, None, None

  train_dataset = datasets.ImageFolder(root=train_dir,
                                       transform=train_transform)
  test_dataset = datasets.ImageFolder(root=test_dir,
                                      transform=test_transform)

  train_dataloader = DataLoader(dataset=train_dataset,
                                batch_size=batch_size,
                                shuffle=True,
                                pin_memory=pin_memory)

  test_dataloader = DataLoader(dataset=test_dataset,
                               batch_size=batch_size,
                               shuffle=False,
                               pin_memory=pin_memory)

  classes = train_dataset.classes
  return train_dataloader, test_dataloader, classes, train_dataset, test_dataset

### `evaluate_model_performance()`

to evaluate the model performance

In [4]:
from timeit import default_timer as timer
from tqdm.auto import tqdm
from typing import Callable, List

import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

try:
  from calflops import calculate_flops
except:
  !pip install calflops
  from calflops import calculate_flops

def evaluate_model_performance(
    model: torch.nn.Module,
    data_loader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    accuracy_fn: Callable,
    class_names: List[str],
    calculate_flops_value: bool,
    display_conf_matrix: bool,
    device: torch.device=device
):
    """
    Evaluates a PyTorch classification model and returns:

    - test loss
    - overall accuracy
    - confusion matrix
    - class-wise accuracy
    - FLOPs (optional)

    Args:
        model: Trained PyTorch model.
        data_loader: Test/validation DataLoader.
        loss_fn: Loss function.
        accuracy_fn: Accuracy function.
        class_names: List of class names in the same order as model classes.
        calculate_flops: Whether to calculate FLOPs.

    Returns:
        Dictionary containing evaluation results.
    """

    ### 1. EVALUATE THE MODEL

    model.eval()
    accuracy_fn.reset()

    total_loss = 0

    # Store predictions and true labels
    all_preds = []
    all_labels = []

    with torch.inference_mode():

        for X, y in tqdm(data_loader):

            X = X.to(device)
            y = y.to(device)

            y_preds = model(X)

            # Extract logits if model returns an object containing logits
            if hasattr(y_preds, "logits"):
                logits = y_preds.logits
            else:
                logits = y_preds

            # Loss
            loss_batch = loss_fn(logits, y)
            total_loss += loss_batch.item()

            # Overall accuracy
            accuracy_fn.update((logits, y))

            # Predictions
            preds = torch.argmax(logits, dim=1)

            # Save predictions and labels
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # Average loss
    loss = total_loss / len(data_loader)

    # Overall accuracy
    accuracy = accuracy_fn.compute()

    # Convert to numpy arrays
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    ### 2. CONFUSION MATRIX
    cm = confusion_matrix(
        all_labels,
        all_preds,
        labels=list(range(len(class_names)))
    )

    # Display confusion matrix by count each sample image
    if display_conf_matrix:

      conf_matrix_count_disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                                      display_labels=class_names)

      fig, ax = plt.subplots(figsize=(8,8))
      conf_matrix_count_disp.plot(ax=ax, cmap="Blues", values_format="d")

      plt.xticks(rotation=45)
      plt.title("Confusion Matrix")
      plt.show()

      conf_matrix_proportion_disp = ConfusionMatrixDisplay.from_predictions(all_labels,
                                                                            all_preds,
                                                                            display_labels=class_names,
                                                                            normalize="true",
                                                                            cmap="Blues",
                                                                            values_format=".2f")
      plt.xticks(rotation=45)
      plt.title("Normalized Confusion Matrix")
      plt.show()

    ### 3. CLASS-WISE ACCURACY
    classwise_accuracy = {}

    for i in range(len(cm)):

        total_actual = cm[i].sum()

        if total_actual > 0:
            class_acc = cm[i, i] / total_actual
        else:
            class_acc = 0.0

        classwise_accuracy[class_names[i]] = float(class_acc)

    ### 4. FLOPs
    flops_result = None
    if calculate_flops_value:
      # Get one batch only to determine input shape
      sample_X, _ = next(iter(data_loader))
      sample_X = sample_X[:1].to(device)

      flops_result = calculate_flops(model=model,
                                     input_shape=tuple(sample_X.shape)) # we can also hardcode the shape as (1, 3, 224, 224)

    return {
        "model_name": model.__class__.__name__,
        "loss": loss,
        "accuracy": accuracy,
        "confusion_matrix": cm,
        "classwise_accuracy": classwise_accuracy,
        "y_true": all_labels,
        "y_pred": all_preds,
        "flops": flops_result,
    }

## 3. code to turn resuable functions to python script

In [ ]:
%%writefile helper_functions.py

import torch
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from tqdm.auto import tqdm
from timeit import default_timer as time
from typing import Callable

# STEP FUNCTIONS FOR TRAINING
def train_step(model: torch.nn.Module,
               loss_fn: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device):
    """Perform training on model trying to learn on data_loader"""
    model.train()
    loss = 0
    accuracy_fn.reset()

    for (X, y) in tqdm(data_loader, desc="Training Batches", leave=False):
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        outputs = model(X)
        y_preds = outputs.logits if hasattr(outputs, "logits") else outputs

        # 2. Calculate loss and update metrics
        train_loss_batch = loss_fn(y_preds, y)
        loss += train_loss_batch.item()
        accuracy_fn.update((y_preds, y))

        # 3. Backward pass and optimization steps
        optimizer.zero_grad()
        train_loss_batch.backward()
        optimizer.step()

    loss /= len(data_loader)
    accuracy = accuracy_fn.compute()

    print(f"Train loss: {loss:.4f} | train accuracy: {accuracy:.4f}")
    return (loss, accuracy)

def test_step(model: torch.nn.Module,
              loss_fn: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              accuracy_fn,
              device: torch.device):
    """Perform testing on model going over data_loader"""
    test_loss = 0
    accuracy_fn.reset()
    model.eval()

    with torch.inference_mode():
        for X_test, y_test in tqdm(data_loader, desc="Testing Batches", leave=False):
            X_test, y_test = X_test.to(device), y_test.to(device)

            # 1. Forward pass
            outputs = model(X_test)
            y_test_preds = outputs.logits if hasattr(outputs, "logits") else outputs

            # 2. Calculate batch loss
            test_loss_batch = loss_fn(y_test_preds, y_test)
            test_loss += test_loss_batch.item()

            # 3. Track metrics
            accuracy_fn.update((y_test_preds, y_test))

        test_loss /= len(data_loader)
        test_acc = accuracy_fn.compute()

    print(f"Test loss: {test_loss:.4f} | test accuracy: {test_acc:.4f}")
    return (test_loss, test_acc)


# PRINT TRAINING TIME
def print_train_time(start: float, end: float, device: torch.device = None):
    """Prints difference between start and end time"""
    total_time = end - start
    print(f"Train time for device {device}: {total_time:.2f} seconds")
    return total_time


# EVALUATE THE MODEL
def eval_model(
    model: torch.nn.Module,
    data_loader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    accuracy_fn,
    device: torch.device
):
    """Evaluates the given model on the target data loader."""
    model.eval()
    accuracy_fn.reset()

    loss = 0.0

    with torch.inference_mode():
        for X, y in tqdm(data_loader, desc="Evaluating"):
            X, y = X.to(device), y.to(device)
            y_preds = model(X)

            logits = y_preds.logits if hasattr(y_preds, 'logits') else y_preds

            loss_batch = loss_fn(logits, y)
            loss += loss_batch.item() # FIX: used .item() here
            accuracy_fn.update((logits, y))

        loss /= len(data_loader)
        accuracy = accuracy_fn.compute()

    if isinstance(accuracy, torch.Tensor):
        accuracy = accuracy.item()

    return {
        "model_name": model.__class__.__name__,
        "loss": loss,
        "accuracy": accuracy
    }

# TRAIN FUNCTION
def train_give_results(model: torch.nn.Module,
                         loss_fn: torch.nn.Module,
                         optimizer: torch.optim.Optimizer,
                         accuracy_fn: Callable,
                         train_dataloader: torch.utils.data.DataLoader,
                         test_dataloader: torch.utils.data.DataLoader,
                         train_step: Callable,
                         test_step: Callable,
                         print_train_time: Callable,
                         lr_scheduler: Callable,
                         patience: int,
                         epochs: int,
                         device: torch.device):

    training_results = {
        "results": [],
        "train_time": 0,
        "model_layers": None,
        "device": device
    }

    # FIX: Changed 'timer()' to the correct imported alias 'time()'
    time_before_train = time()

    patience_counter = 0
    best_loss = float("inf")

    for epoch in tqdm(range(epochs), desc="Total Epochs"):
        print(f"\nepoch: {epoch}--------------")

        ### 1. Training Step
        train_loss, train_acc = train_step(
            model=model,
            loss_fn=loss_fn,
            optimizer=optimizer,
            accuracy_fn=accuracy_fn,
            data_loader=train_dataloader,
            device=device
        )

        ### 2. Testing Step
        test_loss, test_acc = test_step(
            model=model,
            loss_fn=loss_fn,
            data_loader=test_dataloader,
            accuracy_fn=accuracy_fn,
            device=device
        )

        if isinstance(train_acc, torch.Tensor): train_acc = train_acc.item()
        if isinstance(test_acc, torch.Tensor): test_acc = test_acc.item()

        ### 3. Apply lr scheduler
        lr_scheduler.step(test_loss)

        ### 4. Apply early stopping
        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
            print(f"🔥 New best model saved with test loss: {test_loss:.4f}")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"🛑 Early stopping triggered at epoch {epoch}")
            break

        ### 5. SAVE RESULTS
        training_results['results'].append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_loss,
            "test_acc": test_acc,
        })

    time_after_train = time()
    train_time = print_train_time(time_before_train, time_after_train, device=device)

    training_results["train_time"] = train_time
    training_results["device"] = device

    if hasattr(model, "features"):
        training_results["model_layers"] = str(model.features)
    else:
        training_results["model_layers"] = str(type(model))

    return training_results

# just changed the above train_give_resultsfor CosineAnnealingWarmRestarts schedular
def train_give_results_sc(model: torch.nn.Module,
                          loss_fn: torch.nn.Module,
                          optimizer: torch.optim.Optimizer,
                          accuracy_fn: Callable,
                          train_dataloader: torch.utils.data.DataLoader,
                          test_dataloader: torch.utils.data.DataLoader,
                          train_step: Callable,
                          test_step: Callable,
                          print_train_time: Callable,
                          lr_scheduler: Callable,
                          patience: int,
                          epochs: int,
                          device: torch.device):

    ### INITIALIZE RESULTS DICTIONARY
    training_results = {
        "results": [],
        "train_time": 0,
        "model_layers": None,
        "device": device
    }

    # FIX: Changed 'timer()' to the correct imported 'time()' alias
    time_before_train = time()

    # Variables for early stopping
    patience_counter = 0
    best_loss = float("inf")

    for epoch in tqdm(range(epochs), desc="Total Epochs"):
        print(f"\nepoch: {epoch}--------------")

        ### Training Step
        train_loss, train_acc = train_step(
            model=model,
            loss_fn=loss_fn,
            optimizer=optimizer,
            accuracy_fn=accuracy_fn,
            data_loader=train_dataloader,
            device=device
        )

        ### Testing Step
        test_loss, test_acc = test_step(
            model=model,
            loss_fn=loss_fn,
            data_loader=test_dataloader,
            accuracy_fn=accuracy_fn,
            device=device
        )

        # Ensure tensor values are extracted to primitive float metrics
        if isinstance(train_acc, torch.Tensor): train_acc = train_acc.item()
        if isinstance(test_acc, torch.Tensor): test_acc = test_acc.item()

        ### Apply lr scheduler
        # For CosineAnnealingWarmRestarts, we step per epoch (or batch). No loss metric is passed.
        lr_scheduler.step()

        ### SAVE RESULTS
        training_results['results'].append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_loss,
            "test_acc": test_acc,
        })

        ### Apply early stopping
        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
            print(f"🔥 New best model saved with test loss: {test_loss:.4f}")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("🛑 Early stopping triggered")
            break

    # FIX: Changed 'timer()' to 'time()'
    time_after_train = time()

    # FIX: Updated hardcoded "cuda" to the dynamic 'device' parameter
    train_time = print_train_time(time_before_train, time_after_train, device=device)

    ### LOAD THE BEST WEIGHTS BACK IN MEMORY
    model.load_state_dict(torch.load("best_model.pt", weights_only=True))

    ### SAVE METADATA
    training_results["train_time"] = train_time
    training_results["device"] = device

    if hasattr(model, "features"):
        training_results["model_layers"] = str(model.features)
    else:
        training_results["model_layers"] = str(type(model))

    return training_results

# SAVE MODEL STATE IN DRIVE
def save_model_results(model: torch.nn.Module,
                       optimizer: torch.optim.Optimizer | None,
                       epochs: int,
                       results: dict,
                       path: str):
    """
    Saves the training checkpoint (model weights, optimizer states, and curves)
    to a designated folder path. Assumes Drive mounting is done in the notebook.
    [INFO]: To save model state in drive do this before calling this function.

    from google.colab import drive
    drive.mount('/content/drive')
    """
    checkpoint = {
        "epochs": epochs,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "results": results
    }

    # save to specified path
    torch.save(checkpoint, path)
    print(f"✅ Checkpoint successfully saved to: {path}")

# LOAD SAVED CHECKPOINT
def load_saved_checkpoint(model: torch.nn.Module,
                          device: torch.device,
                          optimizer: torch.optim.Optimizer | None,
                          path: str,
                          load_trained_params: bool):
    """
    Loads saved metadata and optionally map-restores trained parameters back
    into memory for testing or resuming training.
    [INFO]: To load from drive do this before calling this function.

    from google.colab import drive
    drive.mount('/content/drive')
    """
    # Map-locate to current execution device context
    checkpoint = torch.load(path, map_location=device, weights_only=False)

    if load_trained_params:
        model.load_state_dict(checkpoint["model_state_dict"])

        if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            print("🔄 Loaded both model states and optimizer states!")
        else:
            print("🔄 Loaded model weights only.")

    return checkpoint

## 4. code to turn modules to python script

#### `get_fer2013_data()`

In [ ]:
%%writefile get_fer2013_data.py

from pathlib import Path
import requests
import zipfile
import os

def get_fer2013_data(destination: str,
                     remove_source: bool):

  data_path = Path("data")
  image_path = data_patmh / destination

  # Download
  if not image_path.is_dir():
    image_path.mkdir(parents=True, exist_ok=True)

  if not (data_path / "fer2013.zip").is_file():
    print(f"zip file not found downloading from kagglehub")
    # command to download fer2013 data from kaggle - https://www.kaggle.com/datasets/msambare/fer2013
    r = requests.get("https://www.kaggle.com/api/v1/datasets/download/msambare/fer2013")

    with open(data_path / "fer2013.zip", "wb") as f:
      f.write(r.content)

    # extract the zip file data into image_path
    with zipfile.ZipFile(data_path / "fer2013.zip", "r") as zip_ref:
      zip_ref.extractall(image_path)

  if remove_source:
    # remove the zip file
    os.remove(data_path / "fer2013.zip")
    print(f"Extracted {data_path/"fer2013.zip"} file to {image_path}")

  return image_path

### `create_dataloaders()`

In [ ]:
%%writefile create_dataloaders.py

import torch
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import os

# NUM_WORKERS = os.cpu_count()

device = "cuda" if torch.cuda.is_available() else "cpu"

PIN_MEMORY = True if device == "cuda" else False

def create_dataloaders(train_dir: str,
                       test_dir: str,
                       batch_size: int,
                       train_transform: transforms.Compose=None,
                       test_transform: transforms.Compose=None,
                      #  num_workers: int=NUM_WORKERS,
                       pin_memory: bool=PIN_MEMORY):
  if train_transform is None or test_transform is None:
    print(f"[INFO] train and test transform both need to be passed. Exitting...")
    return None, None, None

  train_dataset = datasets.ImageFolder(root=train_dir,
                                       transform=train_transform)
  test_dataset = datasets.ImageFolder(root=test_dir,
                                      transform=test_transform)

  train_dataloader = DataLoader(dataset=train_dataset,
                                batch_size=batch_size,
                                shuffle=True,
                                pin_memory=pin_memory)

  test_dataloader = DataLoader(dataset=test_dataset,
                               batch_size=batch_size,
                               shuffle=False,
                               pin_memory=pin_memory)

  classes = train_dataset.classes
  return train_dataloader, test_dataloader, classes, train_dataset, test_dataset

Writing create_dataloaders.py


### `evaluate_model_performance()`

In [8]:
%%writefile evaluate_model_performance.py

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

from timeit import default_timer as timer
from tqdm.auto import tqdm
from typing import Callable, List

import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def evaluate_model_performance(
    model: torch.nn.Module,
    data_loader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    accuracy_fn: Callable,
    class_names: List[str],
    display_conf_matrix: bool,
    calculate_flops: Callable | None=None,
    calculate_flops_value: bool=False,
    device: str=device
):
    """
    Evaluates a PyTorch classification model and returns:

    - test loss
    - overall accuracy
    - confusion matrix
    - class-wise accuracy
    - FLOPs (optional)

    [NOTE]: This function needs calfops library's calculate_flops function if you want to calculate FLOPs.
    You can install by running the following script before calling this function

    try:
      from calflops import calculate_flops
    except:
      !pip install calflops
      from calflops import calculate_flops


    [NOTE]: This function needs accuracy function and if you want pytorch-ignite's accuracy function,
    ignite accuracy_fn supports update, reset functions used in this function
    You can install by running the following script before calling this function

    try:
      from ignite.metrics import Accuracy
    except:
      !pip install pytorch-ignite
      from ignite.metrics import Accuracy
    accuracy_fn = Accuracy()


    Args:
        model: Trained PyTorch model.
        data_loader: Test/validation DataLoader.
        loss_fn: Loss function.
        accuracy_fn: Accuracy function.
        class_names: List of class names in the same order as model classes.
        calculate_flops_value: Whether to calculate FLOPs.
        calculate_flops: to calculate FLOPS

    Returns:
        Dictionary containing evaluation results.
    """

    ### 1. EVALUATE THE MODEL

    model.eval()
    accuracy_fn.reset()

    total_loss = 0

    # Store predictions and true labels
    all_preds = []
    all_labels = []

    with torch.inference_mode():

        for X, y in tqdm(data_loader):

            X = X.to(device)
            y = y.to(device)

            y_preds = model(X)

            # Extract logits if model returns an object containing logits
            if hasattr(y_preds, "logits"):
                logits = y_preds.logits
            else:
                logits = y_preds

            # Loss
            loss_batch = loss_fn(logits, y)
            total_loss += loss_batch.item()

            # Overall accuracy
            accuracy_fn.update((logits, y))

            # Predictions
            preds = torch.argmax(logits, dim=1)

            # Save predictions and labels
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # Average loss
    loss = total_loss / len(data_loader)

    # Overall accuracy
    accuracy = accuracy_fn.compute()

    # Convert to numpy arrays
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    ### 2. CONFUSION MATRIX
    cm = confusion_matrix(
        all_labels,
        all_preds,
        labels=list(range(len(class_names)))
    )

    # Display confusion matrix by count each sample image
    if display_conf_matrix:

      conf_matrix_count_disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                                      display_labels=class_names)

      fig, ax = plt.subplots(figsize=(8,8))
      conf_matrix_count_disp.plot(ax=ax, cmap="Blues", values_format="d")

      plt.xticks(rotation=45)
      plt.title("Confusion Matrix")
      plt.show()

      conf_matrix_proportion_disp = ConfusionMatrixDisplay.from_predictions(all_labels,
                                                                            all_preds,
                                                                            display_labels=class_names,
                                                                            normalize="true",
                                                                            cmap="Blues",
                                                                            values_format=".2f")
      plt.xticks(rotation=45)
      plt.title("Normalized Confusion Matrix")
      plt.show()

    ### 3. CLASS-WISE ACCURACY
    classwise_accuracy = {}

    for i in range(len(cm)):

        total_actual = cm[i].sum()

        if total_actual > 0:
            class_acc = cm[i, i] / total_actual
        else:
            class_acc = 0.0

        classwise_accuracy[class_names[i]] = float(class_acc)

    ### 4. FLOPs
    flops_result = None
    if calculate_flops_value:
      # throw error of calculate_flops is not supplied to function
      if calculate_flops is None:
        raise ValueError(
            "calculate_flops must be provided when "
            "calculate_flops_value=True."
        )

      # Get one batch only to determine input shape
      sample_X, _ = next(iter(data_loader))
      sample_X = sample_X[:1].to(device)

      flops_result = calculate_flops(model=model,
                                     input_shape=tuple(sample_X.shape)) # we can also hardcode the shape as (1, 3, 224, 224)

    return {
        "model_name": model.__class__.__name__,
        "loss": loss,
        "accuracy": accuracy,
        "confusion_matrix": cm,
        "classwise_accuracy": classwise_accuracy,
        "y_true": all_labels,
        "y_pred": all_preds,
        "flops": flops_result,
    }

Writing evaluate_model_performance.py
